# neo4j & 그래프 DB 소개 (참고용 · 선택)

> 📎 **참고용(선택) 노트북**: neo4j는 부트캠프 후반 **온톨로지·Neo4j·GraphRAG** 모듈에서 스키마 설계부터 GraphRAG까지 본격적으로 다시 다룹니다. 이 노트북은 그 전에 "그래프 DB가 무엇인지" 짧게 맛만 보는 선택 자료입니다 — 시간이 없다면 건너뛰어도 이어지는 학습에 지장이 없습니다.

> **오전엔 '의미가 가까운 문서'를 벡터(pgvector)로 찾았습니다. 이번엔 또 다른 AI 전용 데이터 — '관계'를 다루는 그래프 DB neo4j를 맛봅니다.** 데이터를 **노드(사람·사물) + 관계(엣지)** 로 잇고, '친구의 친구'(다중 홉) 같은 관계 질문이 그래프에서 얼마나 자연스러운지 봅니다. 이것이 부트캠프 후반의 **GraphRAG**로 이어집니다.

## 차시 학습 목표
- 관계 중심 데이터(노드 + 관계)와 그래프 DB의 쓰임을 이해한다
- `db-neo4j`에 드라이버로 접속해 Cypher `CREATE`로 노드·관계를 만들고, `MATCH`로 **1홉·다중 홉** 관계를 조회한다
- 관계 질문은 그래프가 강하다 → 후반부 **GraphRAG** 모듈로 연결한다

## 다루는 내용
- 관계 중심 데이터 & 그래프 DB란 (이론)
- Docker neo4j 기동 & Cypher 맛보기 CREATE/MATCH
- GraphRAG 예고 (이론)

> ⚠️ **표기 규약**: **Cypher 결과(노드/관계 수·MATCH 결과)는 확정값**입니다. 컨테이너 **기동 시간은 '예시'**(기기마다 다름)입니다.
>
> 📌 이 차시는 **'소개·맛보기' 수준** — `CREATE`/`MATCH`까지만 다룹니다(`MERGE`·집계·인덱스·GraphRAG·Text2Cypher는 범위 밖이며 부트캠프 후반 Neo4j 모듈 소관).

In [1]:
import os
os.system('chcp 65001')

0

## 환경 준비 — 패키지 설치

Python용 neo4j 드라이버입니다.

In [2]:
# ✅ 포인트: neo4j 공식 Python 드라이버(GraphDatabase.driver로 Bolt 연결).
#   - neo4j          : 파이썬에서 neo4j 그래프 데이터베이스에 접속하고 Cypher 쿼리를 실행하게 해주는 공식 드라이버
#   - python-dotenv  : .env 파일에 적어둔 비밀번호를 안전하게 읽어오는 도구 (PostgreSQL 실습과 같은 습관)
%pip install -q neo4j python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 관계 중심 데이터 & 그래프 DB란 (이론)

지난 시간에 NoSQL 4유형 중 **graph(neo4j)는 오늘 맛본다**고 했죠? 지금입니다.

- **그래프 = 노드 + 관계**: 노드(사람·사물) 사이를 관계(엣지)가 잇습니다. 예: `(앨리스)-[:KNOWS]->(밥)`.
- **언제 그래프인가**: 관계 자체가 중요하고, **다중 홉** 질문이 필요할 때 — SNS 친구, 추천, 지식 그래프.
- **RDBMS와 대비**: '친구의 친구의 친구'는 SQL이라면 **JOIN을 여러 번** 해야 해 번거롭고 느립니다. 그래프는 **화살표를 따라가면** 끝.
- **neo4j·Cypher**: 대표 그래프 DB neo4j는 SQL 대신 **Cypher**라는 언어를 씁니다. `(a)-[:R]->(b)` 같은 ASCII-art 패턴이 직관적입니다.

> **✅ 포인트**: 그래프 = 노드 + 관계, **다중 홉 관계에 강함**. 비유하면 그래프는 '인맥 지도'(누가 누구를 아는가)예요.

## Docker neo4j 기동 & Cypher 맛보기 CREATE/MATCH

아래 셀들이 순서대로 **이미지 확인 → (없으면) 컨테이너 생성 / (있으면) 재사용 → 실행 상태 확인 → 포트 확인**까지 처리합니다 — PostgreSQL 실습(`05_PostgreSQL_1_Docker`)에서 쓴 것과 같은 **멱등(재실행 안전) 패턴**입니다. 최초 실행이든 재실행이든 위에서부터 그대로 실행하면 됩니다.

기동 명령(참고 — 아래 셀이 자동으로 실행합니다):
```
docker run -d --name db-neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/neo4jpass neo4j:2026.06-community
```
- **7474** = 브라우저(그래프 시각화), **7687** = Bolt(드라이버 접속)
- 📌 **볼륨 생략**(그래프는 노트북에서 재생성 — 지난 시간 Redis 볼륨 생략과 같은 원칙). 운영은 볼륨으로 영속화합니다.
- 포트·환경변수(`-e`)의 일반적인 의미는 PostgreSQL 실습에서 다뤘습니다 — 여기서는 neo4j에 맞는 값만 확인합니다.

In [3]:
# ✅ 포인트: 이미지 = 컨테이너를 찍어내는 '틀'. 로컬에 있어야 바로 기동됩니다(없으면 run이 최초 1회 자동 pull).
!docker images neo4j:2026.06-community

IMAGE   ID             DISK USAGE   CONTENT SIZE   EXTRA


In [4]:
# 💡 재실행 : db-neo4j가 이미 있으면 새로 만들지 않고 재사용하고, 없을 때만 새로 만듭니다.
# 아래는 Windows(cmd.exe) 기준입니다 — Linux/macOS는 바로 아래 주석 처리된 줄을 대신 쓰세요.
# 셀 맨 앞의 !는 "이 줄은 파이썬 코드가 아니라 터미널(운영체제) 명령어다"라는 뜻의 Jupyter 표시입니다.
# docker inspect db-neo4j : db-neo4j라는 이름의 컨테이너가 존재하는지 확인합니다(있으면 성공, 없으면 실패).
#   >NUL 2>&1        : 그 확인 과정의 출력·에러 메시지를 화면에 띄우지 않고 숨깁니다.
#   && echo ...      : 존재하면(성공하면) "이미 있음" 메시지만 출력합니다.
#   || docker run ...: 존재하지 않으면(실패하면) 그때 처음으로 새 컨테이너를 만듭니다.
!docker inspect db-neo4j >NUL 2>&1 && echo db-neo4j 컨테이너가 이미 있습니다 - 재사용합니다. || docker run -d --name db-neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/neo4jpass neo4j:2026.06-community
# ---- Linux/macOS ----
# !docker inspect db-neo4j >/dev/null 2>&1 && echo "db-neo4j 컨테이너가 이미 있습니다 - 재사용합니다." || docker run -d --name db-neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/neo4jpass neo4j:2026.06-community

# docker start db-neo4j : 방금 만들었든 예전부터 있었든, 혹시 멈춰 있다면 다시 켭니다(이미 켜져 있으면 아무 일도 안 함).
!docker start db-neo4j >NUL 2>&1 && echo db-neo4j 실행 상태 보장 완료
# ---- Linux/macOS ----
# !docker start db-neo4j >/dev/null 2>&1 && echo "db-neo4j 실행 상태 보장 완료"

470f1c8e37cf7f3b4a8582f07e0dfcb3e9f68af1dfc4573bfc356e6399d4cfcc


Unable to find image 'neo4j:2026.06-community' locally
2026.06-community: Pulling from library/neo4j
4f4fb700ef54: Pulling fs layer
7c7c0e5de1c9: Pulling fs layer
26c307b5e35a: Pulling fs layer
074eb2e29a44: Pulling fs layer
0cce7bb35f0c: Pulling fs layer
4f4fb700ef54: Download complete
0cce7bb35f0c: Download complete
26c307b5e35a: Download complete
ee78585104da: Download complete
26c307b5e35a: Pull complete
3cb6d878e3d6: Download complete
7c7c0e5de1c9: Download complete
0cce7bb35f0c: Pull complete
7c7c0e5de1c9: Pull complete
074eb2e29a44: Download complete
4f4fb700ef54: Pull complete
074eb2e29a44: Pull complete
Digest: sha256:42fd5b9ead4dd4211f6f91bd831c358e4e2117367d04633fbf88682ca4792b30
Status: Downloaded newer image for neo4j:2026.06-community


db-neo4j 실행 상태 보장 완료


In [5]:
# ✅ 포인트: docker ps --filter name=db-neo4j = db-neo4j 컨테이너만 골라 보여줍니다. STATUS가 Up, PORTS에 7474·7687이 보이면 정상입니다.
# ⚠️ Jupyter의 ! 매직에서는 docker --format 의 Go 템플릿({{...}})을 쓰지 않습니다(중괄호가 깨짐 — 상태 확인은 --filter까지만 씁니다).
# (db-redis·db-pg 등 다른 컨테이너까지 함께 보고 싶다면 --filter 없이 docker ps 만 실행해도 됩니다.)
!docker ps --filter name=db-neo4j

CONTAINER ID   IMAGE                     COMMAND                  CREATED          STATUS          PORTS                                                                                      NAMES
470f1c8e37cf   neo4j:2026.06-community   "tini -g -- /startup…"   37 seconds ago   Up 36 seconds   0.0.0.0:7474->7474/tcp, [::]:7474->7474/tcp, 0.0.0.0:7687->7687/tcp, [::]:7687->7687/tcp   db-neo4j


In [6]:
# ✅ 포인트: 포트 매핑 = 컨테이너 안(7474·7687)과 내 PC를 잇는 '출입문'. docker port로 실제 매핑을 확인할 수 있습니다.
!docker port db-neo4j

7474/tcp -> 0.0.0.0:7474
7474/tcp -> [::]:7474
7687/tcp -> 0.0.0.0:7687
7687/tcp -> [::]:7687


### 브라우저로 그래프를 눈으로 (neo4j의 강점)

브라우저에서 **http://localhost:7474** 에 접속하면(로그인 `neo4j` / `neo4jpass`) 노드-관계 그래프를 **눈으로** 볼 수 있습니다. 이 시각화가 neo4j의 큰 강점입니다. 아래에서는 Python 드라이버로 같은 그래프를 만들고 조회합니다.

In [7]:
# ✅ 포인트: neo4j-driver로 Bolt(7687)에 연결합니다. verify_connectivity로 준비 확인 — 기동 직후엔 재시도가 필요할 수 있어 반복 시도로 감쌉니다.
# Bolt는 neo4j 전용 통신 프로토콜입니다(PostgreSQL이 5432 포트를 쓰듯, neo4j는 보통 7687 포트를 씁니다).
from neo4j import GraphDatabase

# 📌 비밀번호는 .env의 NEO4J_PASSWORD에서 읽습니다(하드코딩 금지 — PostgreSQL과 같은 습관).
import os
import time
from dotenv import load_dotenv
load_dotenv()   # 같은 폴더의 .env 파일 → 환경변수로 등록
# GraphDatabase.driver(주소, auth=(아이디, 비밀번호)) 로 접속 정보를 담은 driver 객체를 만듭니다.
# 이 시점에는 아직 실제로 연결을 시도하지 않고, "이렇게 접속할 준비"만 해둔 상태입니다.
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", os.environ["NEO4J_PASSWORD"]))

# 💡 재시도 루프: neo4j는 포트가 열린 뒤에도 내부 초기화가 몇 초 더 걸릴 수 있어, 첫 시도가 바로 실패할 수 있습니다.
#   (PostgreSQL의 pg_isready 같은 별도 확인 도구 대신, 여기서는 실제로 쓸 driver.verify_connectivity() 자체를
#    최대 30번 · 1초 간격으로 반복 시도합니다 — neo4j 버전마다 달라지는 로그 문구를 찾는 것보다 이 방법이 더 확실합니다.)
connected = False   # 아직 연결 성공 못함 표시
for attempt in range(30):
    try:
        driver.verify_connectivity()   # 성공하면 예외 없이 지나갑니다.
        connected = True
        break                          # 성공했으니 더 기다리지 않고 반복문을 바로 빠져나갑니다.
    except Exception:
        time.sleep(1)                  # 아직 준비 안 됐으면 1초 쉬고 다시 시도합니다.

if not connected:
    driver.verify_connectivity()   # 30번(약 30초) 모두 실패했다면 마지막으로 한 번 더 — 진짜 문제라면 원래 에러가 그대로 보입니다.

print("neo4j 연결 완료")

neo4j 연결 완료


### Cypher `CREATE` — 노드 4명 + 관계 4개

사람 4명(앨리스·밥·캐럴·댄)을 노드로 만들고 `KNOWS` 관계로 잇습니다.

```
(앨리스)-[:KNOWS]->(밥)     (밥)-[:KNOWS]->(캐럴)
(캐럴)-[:KNOWS]->(댄)       (앨리스)-[:KNOWS]->(댄)
```

In [8]:
# 💡 멱등 초기화: 기존 노드/관계를 모두 지웁니다(재실행 안전).
# driver.session()은 neo4j와 대화할 "세션"을 하나 엽니다(PostgreSQL의 conn과 비슷한 역할).
# with ... as s: 구문은 이 블록이 끝나면 세션을 자동으로 정리(종료)해줍니다.
with driver.session() as s:
    # Cypher(neo4j의 쿼리 언어)에서 MATCH (n)은 "모든 노드 n을 찾아라"는 뜻입니다.
    # DETACH DELETE n : 찾은 노드 n과, 그 노드에 연결된 관계(엣지)까지 함께 삭제합니다.
    #   (DETACH 없이 노드만 지우려 하면, 관계가 남아있는 노드는 삭제할 수 없어 에러가 납니다.)
    s.run("MATCH (n) DETACH DELETE n")

# ✅ 포인트: CREATE로 사람 4명(노드) + KNOWS 4개(관계)를 만듭니다.
# Cypher의 CREATE는 SQL의 INSERT와 비슷하게 "새로 만들기"를 담당합니다.
#   (a:Person {name:'앨리스'}) : Person이라는 라벨(종류)을 가지고, name 속성이 '앨리스'인 노드를 만들고 a라는 별명을 붙입니다.
#   (a)-[:KNOWS]->(b)          : a에서 b로 향하는 KNOWS(안다)라는 이름의 관계(화살표)를 만듭니다.
# 아래는 사람 4명(앨리스·밥·캐럴·댄)과, 그들 사이의 KNOWS 관계 4개를 한 번에 만드는 Cypher 쿼리입니다.
create_query = """
CREATE (a:Person {name:'앨리스'}),
       (b:Person {name:'밥'}),
       (c:Person {name:'캐럴'}),
       (d:Person {name:'댄'}),
       (a)-[:KNOWS]->(b),
       (b)-[:KNOWS]->(c),
       (c)-[:KNOWS]->(d),
       (a)-[:KNOWS]->(d)
"""
with driver.session() as s:
    s.run(create_query)   # 세션을 통해 위 Cypher 쿼리를 실제로 실행합니다.
print("노드·관계 생성 완료")

노드·관계 생성 완료


In [9]:
# ✅ 포인트: 노드/관계 수 확인 → Person 4개, KNOWS 4개 (확정값).
with driver.session() as s:
    # MATCH (p:Person) RETURN count(p) AS n : Person 라벨을 가진 모든 노드를 찾아 그 개수를 셉니다.
    #   (SQL의 "SELECT COUNT(*) FROM ..." 과 같은 역할이라고 생각하면 됩니다.)
    # .single()은 결과가 한 줄뿐일 때 그 한 줄을 바로 꺼내오고, ["n"]으로 n이라는 이름의 값을 꺼냅니다.
    n = s.run("MATCH (p:Person) RETURN count(p) AS n").single()["n"]
    # ()-[k:KNOWS]->() : 라벨 상관없이, KNOWS 관계로 연결된 모든 화살표를 찾아 그 개수를 셉니다.
    k = s.run("MATCH ()-[k:KNOWS]->() RETURN count(k) AS k").single()["k"]
print(f"Person 노드 수 → {n}, KNOWS 관계 수 → {k}")   # 기대: 4, 4

Person 노드 수 → 4, KNOWS 관계 수 → 4


### Cypher `MATCH` 1홉 — 앨리스가 '직접' 아는 사람

`MATCH`는 그래프에서 패턴을 찾습니다. 앨리스에서 `KNOWS` 화살표를 **한 번** 따라가면 → 확정 결과 **밥·댄**.

In [10]:
# ✅ 포인트: 1홉 — (앨리스)-[:KNOWS]->(f) 패턴. 화살표를 한 번 따라갑니다.
# "1홉(hop)"이란 관계(화살표)를 딱 한 번만 따라가는 것을 말합니다 — 여기서는 "앨리스가 직접 아는 사람"을 찾습니다.
with driver.session() as s:
    rows = s.run(
        # (:Person {name:'앨리스'}) : name이 '앨리스'인 Person 노드를 찾습니다. (별명을 안 붙였으니 이 노드는 다시 안 씀)
        # -[:KNOWS]->(f:Person)     : 그 노드에서 KNOWS 화살표를 따라가 도착하는 Person 노드를 f라는 별명으로 받습니다.
        # RETURN f.name AS name     : f 노드의 name 속성만 결과로 돌려받습니다.
        "MATCH (:Person {name:'앨리스'})-[:KNOWS]->(f:Person) "
        "RETURN f.name AS name ORDER BY f.name"
    ).data()   # .data()는 결과 여러 줄을 파이썬 딕셔너리(dict) 리스트로 한 번에 가져옵니다.
print("앨리스가 아는 사람(1홉) →", [r["name"] for r in rows])   # 기대: ['댄', '밥']

앨리스가 아는 사람(1홉) → ['댄', '밥']


### Cypher `MATCH` 다중 홉 — 앨리스의 '친구의 친구'

이번엔 `KNOWS`를 **두 번** 따라갑니다 — '친구의 친구'. 단, 앨리스 본인과 '직접 친구'는 제외합니다. 확정 결과는 **캐럴**입니다(앨리스→밥→캐럴; 댄은 앨리스가 직접 아는 사람이라 제외).

이 '친구의 친구' 질문이 **SQL JOIN이라면 번거롭지만, Cypher는 화살표를 한 줄 더 이으면 끝**입니다. 관계 질문은 그래프가 강합니다.

In [11]:
# ✅ 포인트: 다중 홉 — KNOWS를 두 번 따라(친구의 친구). 본인·직접 친구는 제외.
# 아래 쿼리는 화살표를 2번 연달아 따라가서 "친구의 친구"를 찾되, 자기 자신과 이미 직접 아는 사람은 뺍니다.
query = """
MATCH (a:Person {name:'앨리스'})-[:KNOWS]->(:Person)-[:KNOWS]->(fof:Person)
WHERE fof.name <> '앨리스' AND NOT (a)-[:KNOWS]->(fof)
RETURN DISTINCT fof.name AS name ORDER BY fof.name
"""
# 위 쿼리 해설:
#   MATCH (a:...)- [:KNOWS] -> (:Person) -[:KNOWS]-> (fof:Person)
#     앨리스(a)에서 KNOWS를 한 번, 그다음 또 KNOWS를 한 번 더 따라가 도착한 노드를 fof(friend of friend)라 부릅니다.
#   WHERE fof.name <> '앨리스'         : fof가 앨리스 자기 자신이면 결과에서 뺍니다. (<>는 "같지 않다")
#   AND NOT (a)-[:KNOWS]->(fof)        : a가 fof를 이미 직접 안다면(1홉 친구라면) 그것도 뺍니다.
#   RETURN DISTINCT fof.name           : 이름이 중복될 수 있으니 DISTINCT로 한 번씩만 나오게 합니다.
with driver.session() as s:
    rows = s.run(query).data()
print("앨리스의 친구의 친구(다중 홉) →", [r["name"] for r in rows])   # 기대: ['캐럴']

앨리스의 친구의 친구(다중 홉) → ['캐럴']


> **✅ 포인트**: `docker run db-neo4j` → `CREATE` 노드/관계 → `MATCH` 다중 홉. '친구의 친구'가 Cypher 한 줄로 나옵니다 — SQL JOIN보다 자연스럽죠. 이게 **관계 데이터의 힘**입니다.

## GraphRAG 예고 (이론)

- **GraphRAG 예고**: 오늘 pgvector로 찾은 것은 **의미가 가까운 문서**였습니다(시맨틱 검색). 그런데 '**A와 관계된 것들**' 같은 관계 질문은 그래프가 강합니다. 이 둘(벡터 검색 + 지식 그래프)을 합쳐 더 똑똑하게 검색하는 것이 **GraphRAG**입니다.
- **이후 심화**: **GraphRAG·Text2Cypher**(자연어를 Cypher로 — Text-to-SQL의 그래프판)는 부트캠프 후반 **온톨로지·Neo4j·GraphRAG** 모듈 소관입니다. 오늘은 '그래프가 무엇이고 왜 관계에 강한지' 맛만 봤습니다.

> 📌 **범위**: 오늘은 `CREATE`/`MATCH` 맛보기까지입니다. GraphRAG·Text2Cypher는 후반부 Neo4j 모듈에서 본격적으로 배웁니다.

### 정리 — "임베딩은 벡터로, 관계는 그래프로"
- **임베딩(의미)** 은 **벡터**로 저장·검색합니다 → **pgvector**(⭐PostgreSQL이 임베딩 DB로 좋은 6가지 이유).
- **관계** 는 **그래프**로 잇습니다 → **neo4j**(노드+관계·다중 홉, 후반부에서 이어집니다).

## (참고) 컨테이너 정리

실습이 끝나면 neo4j 컨테이너를 정리할 수 있습니다(메모리 절약). **`db-pg`는 다른 Day와 공유하므로 정지하지 마세요.**
- 중지: `docker stop db-neo4j` / 재개: `docker start db-neo4j`
- 삭제: `docker rm -f db-neo4j` (볼륨이 없어 그래프는 사라집니다 — 노트북에서 다시 만들면 됩니다)

> 아래 셀은 드라이버 연결만 닫습니다(컨테이너는 그대로 둡니다 — 필요할 때 위 명령으로 정리).

In [ ]:
# ✅ 포인트: 드라이버 연결을 닫습니다(컨테이너 db-neo4j는 그대로 — 정리는 위 docker 명령으로).
driver.close()   # 연결을 닫아 자원을 반납합니다 — 이후 이 driver로는 더 이상 쿼리를 실행할 수 없습니다.
print("driver 종료")